# PDAN Part 3
*James Hart*



Dataset:


In [1]:
# %pip install pandas==2.3.3
# %pip install nltk
# %pip install pyspark
# %pip install numpy
# %pip install scikit-learn
# %pip install tensorflow
# %pip install nbformat
# %pip install plotly
# %pip install PyArrow
# %pip install ElementTree
# %pip install seaborn
# %pip install xgboost

### Spark Setup

The data loading, inspection, cleaning, exploratory summaries, and feature preparation below use PySpark DataFrames.
Only small aggregated result sets are collected to the driver for visualisation; the source dataset is not converted to pandas.


To setup Spark through PySpark certain requirements have to be met, the correct versioning of Python, PySpark, Java and tensorflow packages. With Spark being very specific about which versions are to be installed to work correctly. 

Specific versions I am running:
- Python 3.13.5
- Java 25.0.4
- Pandas 2.3.3

Due to the lack of native gpu support in tensorflow on windows, everything was run using my Ryzen 5 3600 cpu (10years old), this impacted performance quite heavily, therefore certain cells and parameters have been tuned to optimse the overall performance and speed.

In [2]:
# Configure the Python executables used by PySpark.
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable


In [3]:
# Make sure Pyspark works correctly without erroring out in the background
import os
import sys
import importlib.metadata

# version of pandas that runs with Java 25.0.4 and Python 3.13.5
PANDAS_REQUIRED_VERSION = "2.3.3"

try:
    installed_version = importlib.metadata.version("pandas")
    if installed_version != PANDAS_REQUIRED_VERSION:
        raise ImportError(f"Version mismatch: Found {installed_version}")
except importlib.metadata.PackageNotFoundError:
    print(f"Error: pandas is not installed.")
    sys.exit(1)
except ImportError as e:
    print(f"{e}. Run: pip install --upgrade pandas=={PANDAS_REQUIRED_VERSION}")
    sys.exit(1)

import pandas as pd
print(f"Loaded pandas: {pd.__version__}")

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
print()

Loaded pandas: 2.3.3



In [4]:
# Create a PySpark session with additional memory.
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("PDAN8412 Part 2")
    .config("spark.memory.offHeap.enabled", "true")
    .config("spark.memory.offHeap.size", "10g")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .config("spark.driver.maxResultSize", "2g")
    .config("spark.sql.execution.arrow.pyspark.enabled", "false")
    .getOrCreate()
)


## Import Dataset

Extract labels from XML sidecar files without making Spark enumerate thousands of small files

In [5]:
import csv
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from xml.etree import ElementTree

ANNOTATIONS_DIR = Path("dataset/Annotations")
LABELS_CSV = Path("dataset/annotation_labels.csv")

def parse_annotation(annotation_path):
    root = ElementTree.parse(annotation_path).getroot()
    filename = root.findtext("filename")
    labels = sorted({name.text for name in root.findall("./object/name") if name.text})
    if not filename or not labels:
        raise ValueError(f"Missing filename or object labels in {annotation_path}")
    return filename, "|".join(labels)

annotation_paths = sorted(ANNOTATIONS_DIR.glob("*.xml"))
if not annotation_paths:
    raise FileNotFoundError(f"No XML annotations found in {ANNOTATIONS_DIR}")

with ThreadPoolExecutor() as executor:
    annotation_rows = list(executor.map(parse_annotation, annotation_paths))

with LABELS_CSV.open("w", newline="", encoding="utf-8") as labels_file:
    writer = csv.writer(labels_file)
    writer.writerow(["filename", "label"])
    writer.writerows(annotation_rows)

print(f"Wrote {len(annotation_rows):,} annotation rows to {LABELS_CSV}")

Wrote 17,125 annotation rows to dataset\annotation_labels.csv


### Spark

In [ ]:
from pathlib import Path
from pyspark.sql.functions import col, regexp_extract

# Enumerate files with Python to avoid Hadoop's broken Windows NativeIO directory listing.
image_paths = [str(path.resolve()) for path in Path("dataset/JPEGImages").glob("*.jpg")]
image_df = spark.read.format("binaryFile").load(image_paths)

# Extract filename from path as the join key
image_df = image_df.withColumn(
    "filename",
    regexp_extract(col("path"), r"([^/\\]+)\.jpg$", 1)
)

In [12]:
print("Images read:", image_df.count())
image_df.select("path", "length").show(5, truncate=False)

Images read: 17125
+---------------------------------------------------------------------------------------------------------+------+
|path                                                                                                     |length|
+---------------------------------------------------------------------------------------------------------+------+
|file:/C:/Users/fanta/OneDrive/Varsity Work/PGDP 2/PDAN8412/Part 3/work/dataset/JPEGImages/2009_004537.jpg|855251|
|file:/C:/Users/fanta/OneDrive/Varsity Work/PGDP 2/PDAN8412/Part 3/work/dataset/JPEGImages/2011_006230.jpg|782249|
|file:/C:/Users/fanta/OneDrive/Varsity Work/PGDP 2/PDAN8412/Part 3/work/dataset/JPEGImages/2009_001444.jpg|677151|
|file:/C:/Users/fanta/OneDrive/Varsity Work/PGDP 2/PDAN8412/Part 3/work/dataset/JPEGImages/2009_004444.jpg|373871|
|file:/C:/Users/fanta/OneDrive/Varsity Work/PGDP 2/PDAN8412/Part 3/work/dataset/JPEGImages/2012_003706.jpg|333836|
+------------------------------------------------------------

In [13]:
# Load annotation labels and extract the image stem to match image_df.filename.
annotation_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(LABELS_CSV))
    .withColumn(
        "filename",
        regexp_extract(col("filename"), r"([^/\\]+)\.jpg$", 1)
    )
)

# Join image metadata with annotation labels.
image_annotation_df = image_df.join(
    annotation_df,
    on="filename",
    how="inner"
)

print("Joined images:", image_annotation_df.count())
image_annotation_df.select("filename", "label", "length").show(10, truncate=False)

Joined images: 17125
+-----------+------------------------------------+------+
|filename   |label                               |length|
+-----------+------------------------------------+------+
|2009_004537|bottle|chair|diningtable|pottedplant|855251|
|2011_006230|person                              |782249|
|2009_001444|cat                                 |677151|
|2009_004444|bicycle                             |373871|
|2012_003706|person                              |333836|
|2009_002194|bicycle                             |320867|
|2009_003218|cow                                 |304426|
|2008_002866|horse|person                        |300506|
|2012_004075|person                              |294114|
|2008_000647|person                              |291640|
+-----------+------------------------------------+------+
only showing top 10 rows
